### 1.Load 'fest_master.csv' in a notebook named 'fest_reshape.ipynb', assign every row a 'day' of 1, 2 or 3 with `np.random.seed(42)` and `np.random.randint`, and save the rows of each day by boolean filtering as 'fest_day1.csv', 'fest_day2.csv' and 'fest_day3.csv' with `index=False`. Print the three row counts. Hint: if 'fest_master.csv' is missing, build a 150-row table with reg_id, event_name, student_id and ticket_inr.

In [1]:
import numpy as np
import pandas as pd

gym = None
fest_df = pd.read_csv('fest_master.csv')

np.random.seed(42)

fest_df['day'] = np.random.randint(1, 4, len(fest_df))

day1 = fest_df[fest_df['day'] == 1]
day2 = fest_df[fest_df['day'] == 2]
day3 = fest_df[fest_df['day'] == 3]

day1.to_csv('fest_day1.csv', index=False)
day2.to_csv('fest_day2.csv', index=False)
day3.to_csv('fest_day3.csv', index=False)

print("Day 1 rows:", len(day1))
print("Day 2 rows:", len(day2))
print("Day 3 rows:", len(day3))

Day 1 rows: 48
Day 2 rows: 46
Day 3 rows: 56


### 2.Read the three day files and stack them row-wise into 'fest_all' with `pd.concat` and `ignore_index=True`. Check that `fest_all.shape[0]` is 150 and that the index runs from 0 to 149. Constraint: Do not use `DataFrame.append()`; it has been removed from pandas.

In [2]:
day1 = pd.read_csv('fest_day1.csv')
day2 = pd.read_csv('fest_day2.csv')
day3 = pd.read_csv('fest_day3.csv')

fest_all = pd.concat(
    [day1, day2, day3],
    ignore_index=True
)

print(fest_all.head())

print("Total rows:", fest_all.shape[0])
print("Index:", fest_all.index)
print("Index starts:", fest_all.index.min())
print("Index ends:", fest_all.index.max())

   reg_id  student_id  event_id  ticket_inr  attended         college  year  \
0       2          41         7         139       NaN  Modern College   1.0   
1       5          12         5         459       0.0     GIT College   2.0   
2       6          34         4         139       1.0     ABC College   2.0   
3      13          35         4         163       1.0     GIT College   1.0   
4      16           3         7         425       1.0     GIT College   2.0   

         event_name  category       venue  budget_inr  volunteers  day  
0    Football Match    Sports    Ground 2       19785          11    1  
1   Singing Contest  Cultural  Auditorium        6838          17    1  
2  Quiz Competition      Tech      Hall B        9804          15    1  
3  Quiz Competition      Tech      Hall B        9804          15    1  
4    Football Match    Sports    Ground 2       19785          11    1  
Total rows: 150
Index: RangeIndex(start=0, stop=150, step=1)
Index starts: 0
Index ends

### 3.Build 'rev_pivot' from 'fest_all' with `pivot_table`: one row per event_name, one column per day, ticket_inr summed with `aggfunc='sum'`. Print it. Hint: add `fill_value=0` so an event-day pair with no bills shows 0 instead of NaN.

In [3]:
rev_pivot = fest_all.pivot_table(
    index='event_name',
    columns='day',
    values='ticket_inr',
    aggfunc='sum',
    fill_value=0
)

print(rev_pivot)

day                    1     2     3
event_name                          
Coding Competition   440  1130  2304
Cricket Match        747   661  1307
Dance Battle        2361  1822  1720
Drama Competition   1178  1060  1977
Football Match      2041  3101   698
Hackathon           1681   991  2942
Quiz Competition    1124  2548  2673
Singing Contest     2778  1764  1956


### 4.Create 'target_long' with 24 rows (each event_name with day 1 to 3 and a random target_inr in multiples of 500 from 2,000 to 10,000, `np.random.seed(42)`), reshape it with `pivot()` into 'target_pivot', then place 'rev_pivot' and 'target_pivot' side by side with `pd.concat(axis=1)` after `add_prefix('actual_')` and `add_prefix('target_')`. Save as 'fest_target_vs_actual.csv'.

In [4]:
np.random.seed(42)

target_long = pd.DataFrame({
    'event_name': np.repeat(fest_all['event_name'].unique(), 3),
    'day': list(range(1, 4)) * 8,
    'target_inr': np.random.randint(4, 21, 24) * 500
})

target_pivot = target_long.pivot(
    index='event_name',
    columns='day',
    values='target_inr'
)

target_pivot.columns = target_pivot.columns.astype(str)

rev_pivot.columns = rev_pivot.columns.astype(str)

comparison = pd.concat([
    rev_pivot.add_prefix('actual_'),
    target_pivot.add_prefix('target_')
], axis=1)

comparison.to_csv('fest_target_vs_actual.csv')

print(target_long.head())
print(target_pivot)
print(comparison)
print("CSV file saved successfully!")

        event_name  day  target_inr
0   Football Match    1        5000
1   Football Match    2        9000
2   Football Match    3        7000
3  Singing Contest    1        5500
4  Singing Contest    2        5000
day                    1     2      3
event_name                           
Coding Competition  9000  7500   3000
Cricket Match       6500  9500   9000
Dance Battle        4500  2500   2000
Drama Competition   7500  7500  10000
Football Match      5000  9000   7000
Hackathon           3000  2500   7500
Quiz Competition    7000  3500   5500
Singing Contest     5500  5000   7000
day                 actual_1  actual_2  actual_3  target_1  target_2  target_3
event_name                                                                    
Coding Competition       440      1130      2304      9000      7500      3000
Cricket Match            747       661      1307      6500      9500      9000
Dance Battle            2361      1822      1720      4500      2500      2000
Drama Com